# Phase 2: Transformer Model

Building on Phase 1, this notebook trains a patch-based transformer classifier and compares it head-to-head against the CNN baseline on the same test set, under identical training conditions (same optimizer, same schedule, same early-stopping rule — see `src/train.py`).

**Note on scope:** the goal here is not to beat state of the art. If the transformer ties or slightly loses to the baseline, that's a legitimate finding to report honestly — the interesting part of this project is the error analysis in Phase 3, not a leaderboard number.

**Why the baseline is retrained in this notebook:** Phase 1 doesn't persist a checkpoint to disk, so to guarantee both models see the exact same subject split and training regime, both are trained fresh here, back to back, with the same seed.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, f1_score
from torch.utils.data import DataLoader

from src.baseline import CNNBaseline
from src.config import BATCH_SIZE, CASSETTE_DIR, LABEL_NAMES, get_device, set_seed
from src.data import (
    SleepEDFDataset,
    build_epoch_arrays,
    discover_subjects,
    get_subject_split,
)
from src.train import evaluate, train_model
from src.transformer import TransformerClassifier

set_seed()
device = get_device()
print(f"device: {device}")

## 1. Rebuild the Phase 1 subject-level split

Same `get_subject_split` with the same seed as Phase 1, so train/val/test subjects are identical to `01_data_baseline.ipynb`.

In [ ]:
subjects = discover_subjects(CASSETTE_DIR)
subject_ids = [s.subject_id for s in subjects]
train_ids, val_ids, test_ids = get_subject_split(subject_ids)
print(f"train subjects ({len(train_ids)}): {train_ids}")
print(f"val subjects   ({len(val_ids)}): {val_ids}")
print(f"test subjects  ({len(test_ids)}): {test_ids}")

train_epochs, train_labels, _ = build_epoch_arrays(subjects, train_ids)
val_epochs, val_labels, _ = build_epoch_arrays(subjects, val_ids)
test_epochs, test_labels, test_subject_ids = build_epoch_arrays(subjects, test_ids)

train_loader = DataLoader(SleepEDFDataset(train_epochs, train_labels), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SleepEDFDataset(val_epochs, val_labels), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(SleepEDFDataset(test_epochs, test_labels), batch_size=BATCH_SIZE, shuffle=False)

print(f"\ntrain epochs: {train_epochs.shape[0]}")
print(f"val epochs:   {val_epochs.shape[0]}")
print(f"test epochs:  {test_epochs.shape[0]}")

## 2. Retrain the CNN baseline (for a fair, same-run comparison)

In [ ]:
set_seed()
baseline_model = CNNBaseline()
baseline_model, baseline_history = train_model(baseline_model, train_loader, val_loader, device=device)
baseline_results = evaluate(baseline_model, test_loader, device=device)

## 3. Train the transformer

Patch embedding (0.5s patches -> 60 tokens), sinusoidal positional encoding, a stack of transformer encoder layers, and a [CLS] token for classification (see `src/transformer.py` for the pooling-choice justification). Trained with the exact same `train_model` call as the baseline above.

In [ ]:
set_seed()
transformer_model = TransformerClassifier()
n_params = sum(p.numel() for p in transformer_model.parameters())
print(f"transformer parameters: {n_params:,}")

transformer_model, transformer_history = train_model(
    transformer_model, train_loader, val_loader, device=device
)
transformer_results = evaluate(transformer_model, test_loader, device=device)

**Figure: training curves, baseline vs. transformer.** Left: validation loss per epoch for each model — a proxy for how smoothly each optimizes under identical conditions. Right: validation accuracy per epoch. Different final epoch counts are expected since each model triggers early stopping independently.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(baseline_history.val_loss, label="CNN baseline")
ax1.plot(transformer_history.val_loss, label="Transformer")
ax1.set_xlabel("epoch")
ax1.set_ylabel("validation loss")
ax1.set_title("Validation loss")
ax1.legend()

ax2.plot(baseline_history.val_acc, label="CNN baseline")
ax2.plot(transformer_history.val_acc, label="Transformer")
ax2.set_xlabel("epoch")
ax2.set_ylabel("validation accuracy")
ax2.set_title("Validation accuracy")
ax2.legend()

fig.tight_layout()
plt.show()

## 4. Head-to-head comparison on the test set

In [ ]:
def summarize(name, results):
    preds, labels = results["preds"], results["labels"]
    macro_f1 = f1_score(labels, preds, average="macro")
    per_class = f1_score(labels, preds, average=None, labels=list(range(len(LABEL_NAMES))))
    return {"model": name, "macro_f1": macro_f1, **dict(zip(LABEL_NAMES, per_class))}

rows = [
    summarize("CNN baseline", baseline_results),
    summarize("Transformer", transformer_results),
]

header = f"{'model':<14}{'macro F1':>10}" + "".join(f"{name:>8}" for name in LABEL_NAMES)
print(header)
print("-" * len(header))
for row in rows:
    line = f"{row['model']:<14}{row['macro_f1']:>10.4f}" + "".join(f"{row[name]:>8.4f}" for name in LABEL_NAMES)
    print(line)

**Figure: normalized confusion matrices, side by side, same test set.** Rows are true labels, columns predicted, each row sums to 1.0. Compare where the two models' error patterns diverge — identical confusion structure would suggest both are hitting the same information ceiling in the signal itself, rather than one model simply being weaker.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

for ax, name, results in zip(
    axes, ["CNN baseline", "Transformer"], [baseline_results, transformer_results]
):
    cm = confusion_matrix(results["labels"], results["preds"], normalize="true")
    disp = ConfusionMatrixDisplay(cm, display_labels=LABEL_NAMES)
    disp.plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
    ax.set_title(name)

fig.tight_layout()
plt.show()

## Next steps

Whatever the macro F1 gap turns out to be, report it honestly — a tie or a small loss for the transformer is a legitimate result given its much smaller inductive bias toward local patterns compared to a CNN. `notebooks/03_error_analysis.ipynb` is where the project earns its keep: confusion patterns, transition-point errors, subject variability, and confidence calibration, for whichever model (or both) is worth digging into further.